In [13]:
import json
url = "https://www2024.thewebconf.org/"
output_dir = "/data/workspace/cognitive_kernel_GAIA/Files"
token_limit = 64000
html_dir = "/data/workspace/cognitive_kernel_GAIA/HTMLS"

In [14]:
from urllib.parse import urlparse
parsed = urlparse(url)
file_name = parsed.netloc
print(file_name)


www2024.thewebconf.org


In [15]:


# data = [
#     {
#         'task_id': 'test0',
#         "Question": f"""
# Assume you are an **AI assistant specialized in generating guidebooks for websites**. You are given a website, and your task is to generate a **guidebook** for that website and its **subpages**.

# ---

# ### Processing Strategy

# 1. **Maintain a priority queue** that stores subpage URLs to be visited.  
#    - Initially, the queue contains only the given starting URL.  
#    - Maintain a **visited list** to avoid revisiting pages.  
#    - Each time, pick the top URL in the queue to visit.

# 2. **For every webpage visited:**
#    - Scroll through the entire webpage **from top to bottom**.  
#    - Record its basic information according to the **page record format** below.  
#    - Append the record to `guidebook.md` following the save function.

# ---

# ### Page Record Format

# For **every page you visit**, based on the html code of the page, you must write a record in the exact format below:

# ```
# ### Page Record
# - id: the page record number (starting from 0, increment by 1 for each page)

# - url: the current webpage’s URL

# - Golden Path: [root-page -> step-1-page -> ... -> current-page]
#   → Record the complete navigation path from the starting root page to the current page.  
#   → If a button leading to this page belongs to a **drop-down menu**, specify it in parentheses in the path —  
#     for example: `(Calls: Industry)` means the user navigated from the "Calls" drop-down menu by clicking its "Industry" sub-button.  
#   → The initial page is always "root-page".

# - Page Title: [main title or H1 heading of the current page]
#   → The text shown in the browser’s tab or the main heading.

# - Main Content Summary: [detailed description of all visible content sections on this page]
#   → **Describe all main visual and functional sections of the page from top to bottom**, with:
#     1. **Location** (e.g., top navigation bar, left sidebar, main content area, right ad area, footer).
#     2. The **purpose and information** each block provides to users.

# - Page Purpose: [main goal of the current page]
#   → Summarize what the page is mainly designed to do (e.g., “Display company products”, “Provide news updates”, “Offer user login portal”).

# - Interaction Notes: [main interactions available to users]
#   → Describe the major actions a user can take on this page (e.g., “filter by category”, “enter search keywords”, “scroll through image carousel”).
# ```

# ---

# ### Saving Function

# After completing one `page record`, open `guidebook.md` and append it using the following Python code:

# ```python
# from urllib.parse import urlparse
# import os

# url = f"{{url}}"
# domain = urlparse(url).netloc.replace('.', '_')
# output_dir = f"{{output_dir}}"
# output_path = os.path.join(output_dir, f"{{domain}}_guidebook.md")

# os.makedirs(output_dir, exist_ok=True)

# def save_page_record(record_text: str):
#     formatted_entry = f"\n---\n{{record_text.strip()}}\n"
#     with open(output_path, "a", encoding="utf-8") as f:
#         f.write(formatted_entry)
#     print(f"[✔] Saved record to {{output_path}}")
# ```

# ---

# ### Token Limitation

# ```
# Given token limit: {{token_limit}}.
# You must ensure the total number of tokens in "guidebook.md" stays within the limit.
# ```

# **Token-control strategy:**
# - For important pages (especially the homepage and pages within depth ≤ 2 from root), write detailed `page records`.
# - For less important pages (depth ≥ 3), shorten the record or skip it entirely.

# Use the following function to count tokens:

# ```python
# import tiktoken

# def count_tokens(text: str, model: str = "gpt-4o-mini") -> int:
#     encoding = tiktoken.encoding_for_model(model)
#     tokens = encoding.encode(text)
#     return len(tokens)
# ```

# Ensure that the combined token count of the current page record and the existing guidebook content does **not exceed** the given token limit.

# ---

# ### Exploration Rules (for Selecting Subpage Links)

# When browsing each page, determine which links should be added to the exploration queue:

# #### ✅ MUST CLICK (High Priority)

# **1. All links in navigation bars**  
# - *Definition:* “Navigation bars” include top menu bars, main toolbars, side toolbars, headers, and footer navigation.  
# - *Action:* Click **every** link found in these navigation sections。  
# - *If drop-down menus exist:*  
#   - **Do not click the drop-down button itself** (since it only expands submenus and usually doesn’t lead to a new page).  
#   - Instead, **expand the drop-down menu** and **click each sub-button or sublink inside it one by one**.  
#   - Ensure that every individual sublink is recorded and queued for exploration, preserving the hierarchy (e.g., “News Center → Company News”, “News Center → Industry Updates”).  
# - *Examples:*  
#   - Click “Login”, “Register”, “Contact Us” in the top toolbar.  
#   - Click “Home”, “News Center”, “Careers” in the main navigation.  
#   - If a dropdown under “News Center” contains “Company News”, “Industry Updates”:  
#     - Expand the “News Center” dropdown (without clicking the parent “News Center” button).  
#     - Then click the two sublinks “Company News” and “Industry Updates” individually.

# **2. Only click “View More” or summary links in content list areas**
# - *Definition:* “Content list areas” display items such as “News”, “Announcements”, “Products”.
# - *Action:* Only click the **summary/aggregate entrance links** (e.g., “View More”, “More +”, “All News”), often at top-right or bottom of a list.
# - *Do not click individual items* inside these sections.
# - *Examples:*
#   - If a “News” section shows 5 news titles and a “View More” button at the end — add only the “View More” link to the queue.
#   - If a “Products” section shows 3 product cards with a “View All Products” button — add only that aggregate link.

# #### 🚫 DO NOT CLICK (Exclude from Queue)

# 1. **File downloads:** Skip links pointing to `.pdf`, `.zip`, `.exe`, etc.  
# 2. **Forms, logins, or registrations:** Avoid any that require user input or authentication.  
# 3. **External websites:** Do not follow any links pointing to other domains.

# ---

# ### Crawling Workflow

# 1. Start from the given **root URL**, get the html of the page.  
# 2. Scroll through and record all visible content as per the **Page Record Format**.  
# 3. Save the record to `guidebook.md` ensuring tokens stay within the limit.  
# 4. Add eligible subpage links (based on the **exploration rules**) to the priority queue.  
#    - **Recommendation:** Before adding a subpage link into the queue, **open the subpage once to verify that the link is accessible**.  
#    - Record its **actual, valid URL** (after any redirects).  
#    - Then close the subpage immediately to avoid unnecessary resource usage.  
#    - This ensures that all links added to the queue are **genuine, reachable links** rather than placeholder or broken ones.  
#    - **Important:** ⚠️ **Never create or infer links based on your own assumptions or imagination.**  
#      Only use hyperlinks that truly exist and can be clicked on the actual webpage interface.  
# 5. **When selecting the next page to visit:**  
#    - Take the top element (first URL) from the queue.  
#    - **Check if this URL has already been visited**:  
#      - If not visited, open the page.  
#      - Then, **immediately add this URL into the `visited` list**.  
# 6. For each new page, repeat steps 2–5 until you reach the token limit or exhaust the queue.  
# 7. Continue until you reach the token limit or all relevant pages have been explored — **don’t stop early**.

# ---

# ### Expert Trajectory Example 1

# For the site **Jinan University College of Information Science and Technology** (`https://xxxy.jnu.edu.cn/`), the expert process is as follows:

# 1. Scroll through the homepage completely and record detailed page information (as the first and therefore most important page).  
# 2. Save this record to the guidebook.  
# 3. Add subpages to the queue:  
#    - Sequentially add each button and hyperlink found on the visible homepage to the **priority queue**, ensuring that all internal links in the **main navigation bar**, **top toolbar**, and **footer navigation** are included.  
#    - **Before adding a subpage link to the queue**, open the link **once** to verify that it is **real and accessible**, record its **actual resolved URL** (after any redirect if applicable), and then close the tab immediately.  
#    - This step guarantees that every link added to the queue is a **genuine, reachable URL**, not a placeholder or a guessed path.  
#    - ⚠️ **Never create or infer links based on assumptions or imagination** — only use links that truly exist and can be clicked on the actual webpage.  
#    - Specifically, because **"Faculty"** is a **drop-down menu**, **do not click the "Faculty" button itself** (it normally only expands sub-menus and does not navigate).  
#    - Instead, **expand the "Faculty" drop-down** and **individually click and validate** each sub-button inside it (e.g., “Faculty Members,” “High-end Talents”) before adding those verified links to the queue.  
#    - Similarly, if other navigation items (such as “Research,” “Education,” or “Admissions”) contain drop-down menus, follow the same principle: **never click the parent button**, but **open and record all sub-menu links** individually.  
#    - Continue to follow the main exploration rules — only add **aggregate-content entry links** such as “MORE+,” “View All,” or “All News,” but do **not** click specific individual news titles or article links.  
#    - Skip irrelevant or external links (e.g., “JNU Main Site,” “Friendly Links”).  
# 4. **When extracting the next URL from the queue:**  
#    - Check whether this page URL has already been visited.  
#    - If not, open it, and after the page finishes loading, **immediately add this URL to the `visited` list** to prevent duplication.  
# 5. Continue crawling each queued subpage in sequence, writing and appending their page records according to the **Page Record Format**, until you reach the token limit or the queue is exhausted.

# ---

# Example of the **actual page record** for this site’s homepage:

# ```
# ---
# - id: 0

# - url: https://xxxy.jnu.edu.cn/

# - Golden Path: [root-page]
#   → 已从root-page（起始页）直接访问“暨南大学信息科学技术学院”主页，无中间分支。

# - Page Title: 暨南大学信息科学技术学院
#   → 页面浏览器标签及主要H1标题均为“暨南大学信息科学技术学院”。

# - Main Content Summary:
#   → 页面自上而下布局如下：
#     1. 顶部工具栏（页面最上方）：
#       - 包含“暨大主页”、“联系我们”等快捷入口，通常便于访问学校主站及联系信息。
#     2. 主导航栏（顶部主菜单横栏）：
#       - 包含主要栏目，结构有：首页、学院概况、师资力量、人才培养、科学研究、人才招聘、学生园地、党群之窗、招生就业等栏目，部分或全部栏目下设下拉菜单可打开子链接。
#     3. 内容区（页面中央大区）：
#       - “人才引进”专栏：重点展示学院最新人才招聘或引进信息，汇总链接（MORE+）及若干条目链接。
#       - “学院动态”板块：展示最新新闻、成果展示（多个条目链接），右侧有MORE+汇总入口及分页控件。
#       - “通知公告”板块：以时间顺序通告学术讲座、招聘等通知，包含多条内容链接及MORE+汇总入口，分页控件可浏览往期内容。
#       - “党群动态”板块：发布党建/党务活动新闻，含MORE+以及多条具体内容链接。
#       - “在线咨询”一条：常置于内容区底部，便捷跳转沟通入口。
#     4. 页脚（页面底部）：
#       - 通常包括学校地址、联系方式、版权信息等（未在当前树中明细，但常规预设）。
#     5. 其他区块：
#       - 若有侧边栏工具，以及浮动在线服务、返回顶部按钮等，也属可见辅助区块（Accessibility Tree未显示此类区块，视实际展现而定）。

# - Page Purpose:
#   → 用于展示暨南大学信息科学技术学院整体概况、最新动态、人才引进、学术活动、招生就业及党群新闻，为师生及访客提供权威资讯与互动入口。

# - Interaction Notes:
#   → 用户主要可执行操作包括：
#     1. 顶部工具栏可点链接：
#        - 暨大主页、联系我们。
#     2. 主导航栏所有栏目均可点击，且部分栏目的下拉菜单包含子栏目可点链接：
#        - 首页、学院概况、师资力量、人才培养、科学研究、人才招聘、学生园地、党群之窗、招生就业等（下拉子链接需逐项展开获取，但主项皆可点击）。
#     3. 内容区所有板块的“MORE+”链接（汇总入口）、以及各条新闻/通知/动态/讲座等条目链接。
#     4. 内容列表区如分页控件可进行浏览、跳转。
#     5. 内容区“在线咨询”链接可快速跳转至在线沟通页面。
#     6. 其他辅助交互（如页脚或侧边工具栏链接），按页面具体展现补充。
# ---
# ```

# ---

# ### Expert Trajectory Example 2

# For another case, **The Web Conference 2025 website** (`https://www2025.thewebconf.org/`):

# 1. Scroll through and thoroughly document the homepage, covering all visible sections and interactive elements.  
# 2. Save this homepage record to the guidebook.  
# 3. Add subpages to the queue:  
#    - Sequentially collect and add **all clickable items on the top navigation bar** (e.g., “About,” “Calls,” “Accepted,” “Program,” “Venue”), while **skipping external icons or links** (such as “Facebook,” “LinkedIn,” “X/Twitter,” or sponsor pages leading outside the domain).  
#    - **Before adding a subpage link to the queue**, open the subpage once to **verify the link is real and accessible**, record its **actual resolved URL**, and then close it immediately.  
#    - ⚠️ **Never fabricate or guess a link** — only queue links that truly exist on the live webpage and can be clicked directly.  
#    - If any navigation item is a **drop-down menu** (e.g., under “Calls” or “Program”), **do not click the parent button** itself. Instead, expand the dropdown and **individually open, verify, and record** each sublink such as “Calls: Industry Track,” “Calls: Research Track,” or “Program: Keynotes.” Add each verified link into the queue.  
# 4. **When taking the next URL from the queue:**  
#    - Check whether this URL has already been visited.  
#    - If not, open it, and **immediately add it to the `visited` list** before continuing exploration.  
# 5. Open and document each queued subpage, following the same **Page Record Format** and token-control method as with the homepage.  
# 6. Continue exploring sequentially until either the token limit is reached or all valid, accessible subpages have been visited — **do not stop early**.
# ---

# Finally, you will be given a specific URL `{url}`, the token limit `{token_limit}`, and the output directory `{output_dir}`.  
# **Your task:** Generate the complete guidebook for that website based on the instructions above.
# """
#     }
# ]


In [16]:

data = [
    {
        'task_id': 'test0',
        "Question": f"""
Assume you are an **AI assistant specialized in generating guidebooks for websites**.  
You are given a website, and your task is to generate a **guidebook** for that website and its **subpages**.

---

### Processing Strategy

1. **Maintain a priority queue** that stores subpage URLs to be visited.  
   - Initially, the queue contains only the given starting URL.  
   - Maintain a **visited list** to avoid revisiting pages.  
   - Each time, pick the top URL in the queue to visit.

2. **When visiting a webpage:**
   - First, **obtain the HTML source code** of the page.  
   - Then, based **solely on the HTML content**, perform all subsequent steps — including scrolling through virtual content, analyzing visible structure, summarizing sections, identifying valid subpage links, and composing the **Page Record**.

3. **For every webpage visited:**
   - Interpret the HTML as if you visually scroll through the page from top to bottom.  
   - Record its basic information following the **Page Record Format** below.  
   - Append this record to `guidebook.md` using the provided save function.

---

Here’s the same content reorganized into a **consistent hierarchical Markdown structure**, where the entire document is treated as part of a single **Page Record** (`###` heading level), and all other sections use subordinate heading levels (`####`, `#####`, etc.) to clearly express the nested structure.

---

### Page Record

#### Basic Information

- **id:**  
  The page record number (starting from 0, increment by 1 for each page).

- **url:**  
  The current webpage’s full URL.

- **Golden Path:** [root-page -> step-1-page -> ... -> current-page]  
  → Record the complete navigation path from the root page to the current page.  
  → If the page is opened via a button under a **drop-down menu**, specify it in parentheses.  
    Example: “(Admissions: Graduate)” means the sub-button *Graduate* inside the *Admissions* menu.  
  → The initial page is always `"root-page"`.

- **Page Title:**  
  The main title text or H1 heading of the current page.  
  → Extract the visible `<title>` or the main page heading (H1).

- **Page Purpose:**  
  Describe what the page is primarily designed to do.  
  → Example: “Display research updates,” “Provide admissions information,” “List faculty members,” “Offer online consultation access.”


#### Structural Overview & Section Purposes

Describe the **overall structure** of the webpage and the **purpose of each main section**, presented **from top to bottom**.

##### a. Header

- **Location:**  
  The uppermost portion of the page.

- **Purpose & Information:**  
  Summarize what appears here — e.g., site logo, school or site name, institutional branding, or quick access links (such as “University Home,” “Contact,” or “Language Switch”).  
  Explain the purpose (branding, identification, global navigation shortcuts).



##### b. Navibar (Main Navigation Bar)

- **Location:**  
  Directly below the header or at the top horizontal area.

- **Purpose & Information:**  
  Describe the main navigation menu items and their functions — e.g., *Home*, *About*, *Faculty*, *Research*, *Admissions*, *News*.  
  Mention whether it includes submenus or drop-down lists and what information users can access via each.


##### c. Search Box (if present)

- **Location:**  
  Typically in the header or near the navibar.

- **Purpose & Information:**  
  Explain what users can search (articles, news, people, documents, etc.), and how it contributes to information accessibility.


##### d. Main Content ⚠️ *(Most Important Section)*

- **Location:**  
  Central part of the page, below the navigation area.

- **Purpose & Information:**  
  This is the **core** of the webpage. Provide a **detailed summary** of the main visible sections contained within it.

###### Internal Sections (Example Format)

For each internal section:

- **Section Title:**  
  Identify the visible heading or logical label (e.g., *News Section*, *Announcements*, *Research Highlights*, *Events*, *Online Consultation*).

- **Section Content Summary:**  
  Provide a **comprehensive summary** that includes:  
  1. **The main types of content included** — identify and briefly summarize what each content type presents (e.g., news updates, event schedules, research highlights, visual banners, or key announcements). Describe each content’s **core message or focus** in your own words, rather than copying text from the section.  
  2. **The organizational structure** — explain how these contents are displayed and arranged (e.g., in cards, grids, columns, sliders, or lists, with or without “MORE+” buttons or pagination).  
  3. Any **visual or interactive features** observed (e.g., hover effects, clickable cards, carousels, responsive layouts). 


##### e. Footer

- **Location:**  
  Bottom of the page.

- **Purpose & Information:**  
  Include standard institutional details such as address, contact phone and email, copyright, ICP registration number, and sometimes quick links to affiliated institutions or QR codes.  
  Explain the informational role of this section (e.g., official identity, contact access, compliance information).


#### Interactive Elements

List **all interactive components** of the page.  
Describe them **by module or section**, rather than every individual link, unless a specific one has special meaning.

For each module (Header, Navibar, Main Content Sections, Footer):

- Identify **the types of interactive elements** present (e.g., navigation buttons, “MORE+” links, pagination controls, collapsible sections, dropdown menus, online forms).  
- If a section contains many repeated item links (e.g., lists of articles, news items, or events), describe them **in general terms** rather than individually.  
  → Example: “Each news headline in the News Section is clickable and opens a full news article in a new page.”  
- Mention any functional interactions (e.g., “hover to expand,” “submit form,” “scroll to view more,” “redirect to recruitment details”).


---
### HTML Obtaining Function
```python
import requests

url = f"{{url}}"
response = requests.get(url, timeout=100)
response.encoding = response.apparent_encoding
html = response.text
with open(f"{{html_dir}}/{{url.replace('/', '_')}}.txt", "w", encoding="utf-8") as f:
    f.write(html)
```

---

### Saving Function

After completing one `page record`, append it to `guidebook.md` using this Python code:

```python
from urllib.parse import urlparse
import os

url = f"{{url}}"
domain = urlparse(url).netloc.replace('.', '_')
output_dir = f"{{output_dir}}"
output_path = os.path.join(output_dir, f"{{domain}}_guidebook.md")

os.makedirs(output_dir, exist_ok=True)

def save_page_record(record_text: str):
    formatted_entry = f"\n---\n{{record_text.strip()}}\n"
    with open(output_path, "a", encoding="utf-8") as f:
        f.write(formatted_entry)
    print(f"[✔] Saved record to {{output_path}}")
```

---

### Token Limitation

```
Given token limit: {{token_limit}}.
Ensure the total number of tokens in "guidebook.md" stays within this limit.
```

**Token-control strategy:**
- For **important or low-depth pages (depth ≤ 2)**, record detailed content.
- For deeper or less critical pages (depth ≥ 3), shorten summaries or skip them.
- Use this token counter:

```python
import tiktoken

def count_tokens(text: str, model: str = "gpt-4o-mini") -> int:
    encoding = tiktoken.encoding_for_model(model)
    tokens = encoding.encode(text)
    return len(tokens)
```

Ensure that the combined token count of the current page record and existing guidebook content does **not exceed** the given token limit.

---

### Exploration Rules (for Selecting Subpage Links)

When parsing each page’s HTML, identify which links qualify for exploration according to the rules below:

#### ✅ MUST CLICK (High Priority)

**1. All links in navigation bars**  
- Includes top menu bars, main toolbars, sidebars, headers, and footer navigation.  
- Click and record **every** internal link found there.  
- For drop-down menus:  
  - **Do not click the drop-down button itself** (it usually doesn’t lead anywhere).  
  - Instead, **expand the submenu** in the HTML structure and **add each valid sublink individually**.  
- *Example:*  
  - Record “Home”, “About”, “Research”, “Contact Us”, etc.  
  - For menus like “Calls → Industry Track / Research Track”, add both verified sublinks.

**2. “View More”, “More +”, “All News” type links**  
- Found in aggregated list areas such as “News”, “Announcements”, “Publications”.  
- **Only add summary-entry links**, not individual list items.  
- Example:  
  - Add “View All News” but not each news title link.

#### 🚫 DO NOT CLICK

1. File downloads (`.pdf`, `.zip`, `.exe`, etc.).  
2. Forms or logins requiring authentication.  
3. External links pointing to other domains.

---

### Crawling Workflow

1. Start from the given **root URL**.  
2. **Obtain the HTML code** of the current page.  
3. Parse the HTML and scroll through virtually to collect all visible content.  
4. Record the page using the **Page Record Format**.  
5. Save the record to `guidebook.md`, ensuring token count is within limit.  
6. Identify and **add eligible subpage links** (based on exploration rules) to the priority queue.  
   - ⚠️ **Never fabricate or infer links** — only use links that truly appear in the HTML and can be clicked.  
7. **When selecting the next page to visit:**  
   - Take the top (first) URL from the queue.  
   - Check if it’s already visited:  
     - If not, open the page and immediately add it to the `visited` list.  
8. Repeat steps 2–7 until the token limit or queue is exhausted.  
9. Continue until you reach the token limit or all relevant pages have been explored — **do not stop early**.

---



### Expert Trajectory Example 1

For the official website of **The Web Conference 2024** (`https://www2024.thewebconf.org/`):

1. **Homepage Retrieval and Parsing**  
   - Retrieve the HTML source of the homepage.  
   - Parse and document all identifiable sections in top–to–bottom order.  
   - Save this as the **first and primary page record**.

2. **Subpage Collection**  
   - Extract all **clickable internal links** appearing in navigation bars and footers.  
   - **Do not create or infer links manually.**

   **For hierarchical (drop-down) menus:**  
   - Ignore the parent button (e.g., “Calls”) and record only its sub-items (e.g., “Calls → Research Tracks”).  
   - Apply the same rule to all multi-level menu structures.

   **For list sections:**  
   - Only include general or summary links such as “MORE+” or “View All.”  
   - Exclude individual article or item links.

   **Exclude** all irrelevant or external URLs (e.g., “OpenReview,” “Facebook,” etc.).

3. **Exploration Queue Process**  
   For each link in the exploration queue:
   - **Check for duplicates:** Skip if already visited.  
   - **If new:**  
     - Retrieve and parse its HTML.  
     - Record the page structure and metadata.  
     - Identify all valid **internal URLs** (verify accessibility).  
     - Append these discovered links to the exploration queue.  
   - Mark the current page as **visited** once processed.

4. **Continuation Rule**  
   - Continue the above process sequentially until the **token** or **queue** limit is reached.

---

# Example of the **actual page record** for a subpage of this site:

```
---
### Page Record

#### Basic Information

- **id:** 6  
- **url:** https://www2024.thewebconf.org/calls/call-for-papers  
- **Golden Path:** [root-page → (Calls: Research Tracks)]  
- **Page Title:** International World Wide Web Conference 2024 (WWW2024 | The Web Conf 2024) | Call for Papers  
- **Page Purpose:**  
  This page is the **Call for Papers** hub for *The Web Conference 2024* (formerly known as WWW).  
  Its primary purpose is to announce detailed information regarding paper submission: research track themes, deadlines, submission process, formatting requirements, and review policies—serving as an essential reference for researchers and practitioners preparing to submit.

#### Structural Overview & Section Purposes

##### a. Header

- **Location:** Fixed at the top of the page (`fixed-top`).  
- **Purpose & Information:**  
  Contains the official conference logo (linking to the homepage at https://www2024.thewebconf.org) and the main navigation bar.  
  It uses a responsive Bootstrap layout with a collapsible “hamburger” menu on mobile view.  
  This section establishes the site’s identity and provides global access to all primary conference pages.


##### b. Navibar (Main Navigation Bar)

- **Location:** Directly beneath the header.  
- **Purpose & Information:**  
  The navigation menu includes several dropdown categories covering all major areas of the conference site:

  - **About:** General introduction, logo, code of conduct, newsletters.  
  - **Calls:** Sections for Research Tracks, Industry, Workshops, Tutorials, Web4Good, PhD Symposium, etc.  
  - **Accepted:** Lists of accepted papers for all tracks.  
  - **Program:** Schedule Overview, Full Schedule, Keynotes, Awards, Special Days, and Social Events.  
  - **Attending:** Topics on registration, venue, accommodation, check-in, visa information, and presentation instructions.  
  - **Organization:** Organizing committees and review teams.  
  - **Sponsors:** Sponsorship information and partner listings.  
  - **Media:** Video playlists, ACM Digital Library links, photos, and press releases.

  Drop-down submenus provide structured access to different sections of the site, with some external links opening in new browser tabs.

##### c. Search Box

- **Location:** No dedicated search bar is visible in this specific page.  
- **Purpose:** Not applicable on this page; navigation is done through menu links.


##### d. Main Content ⚠️ *(Core Section)*

- **Location:** Central content area below navigation bar (`<main class="container mx-auto">`).  
- **Purpose & Information:**  
  This is the core information zone presenting **the full Call for Papers**, including topics, timelines, submission guidelines, research tracks, formatting rules, and publication policies.  
  The structure relies heavily on Bootstrap accordion components to reveal detailed information by track.

###### ▪ Page Header and Introductory Section
- **Heading:** “CALL FOR PAPERS”  
- **Summary:**  
  States the conference date and location (May 13–17, 2024, Singapore) and introduces the conference’s mission — to investigate the state and evolution of the Web across computer science, economics, and social sciences.

###### ▪ **Important Dates**
- Lists deadlines for abstract, full paper, reviewer discussion period, and notification.  
- Notes that all deadlines follow the “Anywhere on Earth (AoE)” timezone.

###### ▪ **Submission Site**
- Specifies that **OpenReview** will be used for submission and review management.  
- All authors must maintain updated OpenReview profiles for conflict-of-interest matching.  
- Includes links to create a profile and to the submission portal.

###### ▪ **Scope**
- Defines the overall research scope of the conference, emphasizing Web science as an independent yet interdisciplinary field.  
- Displays an **accordion list of research tracks**, including:

  - *Economics, Online Markets, and Human Computation*  
  - *Graph Algorithms and Modeling for the Web*  
  - *Responsible Web*  
  - *Search*  
  - *Security*  
  - *Semantics and Knowledge*  
  - *Social Networks, Social Media, and Society*  
  - *Systems and Infrastructure for Web, Mobile, and WoT*  
  - *User Modeling and Recommendation*  
  - *Web Mining and Content Analysis*  

  Each track shows:
  - Research focus and related topics.  
  - Senior Area Chairs (with names, institutions, and profile links).  
  - Track-specific contact emails (e.g., `TheWebConf24-Econ@ACM.org`).

###### ▪ **Relevance**
- Emphasizes that each submission must clearly relate to the Web; otherwise, it will be desk-rejected.

###### ▪ **Submission Guidelines**
- Outlines rules for:
  - Strict compliance with deadlines.  
  - Author policies — maximum nine submissions per author, no author reordering after submission.  
  - **Double-blind review** procedure.  
  - **Formatting:** ACM template, 8-page limit (+ references/appendix).  
  - **Originality & plagiarism** restrictions.  
  - Ethical use of data and compliance with ACM research policies.  
  - Non-compliance results in automatic desk rejection.

###### ▪ **Reviewing Process**
- Explains that each paper receives at least three reviews under a track-specific Area Chair.  
- Mentions **author rebuttal phase** for addressing reviewer concerns.  
- Final acceptance decisions are made by Program Committee Chairs.

###### ▪ **Conflict of Interest Policy**
- Requires authors and reviewers to declare institutional and personal COIs per ACM guidelines.

###### ▪ **Publication & Presentation Policies**
- All accepted papers:
  - Limited to 8 pages of main content (+ references).  
  - Must complete at least one full registration for inclusion.  
  - Must be presented (oral or poster).  
  - Are encouraged to publicly release artifacts/code with ACM “Artifacts Available” badges.

###### ▪ **Program Committee Co-Chairs**
- Lists the two main program co-chairs:
  - Ravi Kumar (Google)  
  - Hady W. Lauw (Singapore Management University)  
- Contact: `TheWebConf24-pcchairs@ACM.org`


##### e. Footer

- **Location:** Bottom of the page.  
- **Purpose & Information:**  
  - Includes icons linking to **LinkedIn**, **Facebook**, and **Twitter** accounts.  
  - States that *The Web Conference 2024* is organized by the **School of Computing and Information Systems, Singapore Management University (SMU)**.  
  - Contains a hidden maintenance notice section for server updates.  
  - Reinforces the official identity and communication channels of the conference.

#### Interactive Elements

- **Header and Navigation:**
  - Dropdown menus expand/collapse on hover or click.
  - All sublinks redirect to track or section pages.
  - External links (e.g., YouTube, ACM DL, Whova) open in new tabs.
- **Accordion Sections (Main Content):**
  - Clickable headers expand hidden content for each research track.
- **Body Hyperlinks:**
  - Multiple external references (OpenReview, ACM policies, templates).
- **Footer:**
  - Social media icons are interactive external links.
- **Mobile Interactions:**
  - Collapsible hamburger menu adapts navigation layout for smaller screens.

---


```

Finally, you will be given a specific URL `{url}`, the token limit `{token_limit}`, the output directory `{output_dir}`, and the HTML directory `{html_dir}`. 
**Your task:** Generate the complete guidebook for that website **based on its HTML structure** and all the detailed instructions above.
"""
    }
]


In [17]:
print(data[0]['Question'])


Assume you are an **AI assistant specialized in generating guidebooks for websites**.  
You are given a website, and your task is to generate a **guidebook** for that website and its **subpages**.

---

### Processing Strategy

1. **Maintain a priority queue** that stores subpage URLs to be visited.  
   - Initially, the queue contains only the given starting URL.  
   - Maintain a **visited list** to avoid revisiting pages.  
   - Each time, pick the top URL in the queue to visit.

2. **When visiting a webpage:**
   - First, **obtain the HTML source code** of the page.  
   - Then, based **solely on the HTML content**, perform all subsequent steps — including scrolling through virtual content, analyzing visible structure, summarizing sections, identifying valid subpage links, and composing the **Page Record**.

3. **For every webpage visited:**
   - Interpret the HTML as if you visually scroll through the page from top to bottom.  
   - Record its basic information following the **Pa

In [18]:
import json

with open(f'/data/workspace/cognitive_kernel_GAIA/questions/{file_name}_notebook.jsonl', 'w', encoding='utf-8') as f:
    for item in data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')